# Load Inference Results JSON → Silver Table

Reads `inference_results_silver.json` (output from `Inference-GLCReconV9` cell 20)
and writes it to `demo.silver_layer.perception_results`.

**Run this on Databricks** where `spark` is available.

In [ ]:
# %% Config
from pyspark.sql import functions as F

CATALOG     = "demo"
SILVER_TABLE = f"{CATALOG}.silver_layer.perception_results"

# Adjust path to where cells 1-18 of Inference-GLCReconV9 generated it
JSON_PATH = "dbfs:/FileStore/inference_results_silver.json"

print(f"Silver table: {SILVER_TABLE}")
print(f"JSON source : {JSON_PATH}")

In [ ]:
# %% Read JSON into Spark DataFrame
df = spark.read.json(JSON_PATH, multiLine=True)

print(f"Rows loaded: {df.count()}")
print(f"Columns: {df.columns}")
df.printSchema()
df.show(5, truncate=40)

In [ ]:
# %% HOTFIX: Compute incremental observed_at from row ordering
# The inference JSON lacks observed_at. We derive it from robot_code grouping
# with deterministic 70-second spacing (64s cycle + 6s sleep) per robot.
#
# Timeline logic:
#   - Each robot's cycles are 70 seconds apart within a day
#   - Robots are staggered by days: R-00 starts day 0, R-01 day 7, etc.
#   - This ensures the simulation timeline spans multiple distinct hours.

from datetime import datetime, timedelta
from pyspark.sql import Row as SparkRow
from pyspark.sql.types import TimestampType

# ── Collect all rows and compute observed_at ────────────────────────────────
all_rows = df.collect()
print(f"Collected {len(all_rows)} rows for observed_at assignment")

# Base time: 30 days ago from now, midnight UTC
base_time = datetime(2025, 7, 1, 6, 0, 0)  # 2025-07-01 06:00:00

# Group rows by robot_code preserving original order within each group
robot_rows: dict[str, list] = {}
for row in all_rows:
    rc = row.robot_code
    if rc not in robot_rows:
        robot_rows[rc] = []
    robot_rows[rc].append(row)

# Stagger robots by days (each robot gets a different start day offset)
robot_list = sorted(robot_rows.keys())
STAGGER_DAYS = 7  # days between robot start times
CYCLE_SPACING_SEC = 70  # 64s cycle + 6s sleep

rows_with_time = []
for robot_idx, robot_code in enumerate(robot_list):
    rows_for_robot = robot_rows[robot_code]
    # Each robot starts STAGGER_DAYS after the previous one
    robot_base = base_time + timedelta(days=robot_idx * STAGGER_DAYS)

    for cycle_idx, row in enumerate(rows_for_robot):
        # Advance 70 seconds per cycle within the robot's operating day
        observed_at = robot_base + timedelta(seconds=cycle_idx * CYCLE_SPACING_SEC)
        row_dict = row.asDict()
        row_dict["observed_at"] = observed_at
        rows_with_time.append(row_dict)

print(f"Assigned incremental observed_at to {len(rows_with_time)} rows")
print(f"  Robots: {robot_list}")
print(f"  First observed_at: {rows_with_time[0]['observed_at']}")
print(f"  Last observed_at:  {rows_with_time[-1]['observed_at']}")

# ── Reconstruct DataFrame with observed_at ──────────────────────────────────
spark_rows = [SparkRow(**r) for r in rows_with_time]
df = spark.createDataFrame(spark_rows)
print(f"  Schema now includes observed_at: {'observed_at' in df.columns}")
df.select("robot_code", "observed_at").show(10, truncate=False)

In [ ]:
# %% Create silver table and write
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver_layer")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    event_id          STRING,
    robot_code        STRING,
    program_number    BIGINT,
    observed_at       TIMESTAMP,
    abnormal_type     STRING,
    abnormal_score    DOUBLE,
    severity_hint     STRING,
    is_file_anomaly   BOOLEAN,
    per_sensor_scores STRING
)
USING DELTA
""")

# Drop and re-populate
spark.sql(f"DELETE FROM {SILVER_TABLE}")

# HOTFIX: Generate unique event_id from robot_code + observed_at
# (the original JSON had event_id = start_time, which was the same for all rows)
df_clean = df.select(
    F.concat(F.col("robot_code"), F.lit("-"), F.col("observed_at").cast("string")).alias("event_id"),
    "robot_code",
    "program_number",
    "observed_at",
    "abnormal_type",
    "abnormal_score",
    "severity_hint",
    "is_file_anomaly",
    F.to_json("per_sensor_scores").alias("per_sensor_scores"),
)

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

cnt = spark.table(SILVER_TABLE).count()
print(f"  \u2713 {cnt} rows written to {SILVER_TABLE}")

In [ ]:
# %% Verify
print("=== Robots summary ===")
spark.sql(f"""
SELECT robot_code,
       COUNT(*) AS total,
       SUM(CASE WHEN is_file_anomaly THEN 1 ELSE 0 END) AS anomalies,
       MIN(observed_at) AS first_event,
       MAX(observed_at) AS last_event
FROM {SILVER_TABLE}
GROUP BY robot_code
ORDER BY robot_code
""").show(truncate=False)

# Confirm timeline spans multiple distinct hours (critical for simulation)
print("=== Timeline span ===")
spark.sql(f"""
SELECT
    MIN(observed_at) AS timeline_start,
    MAX(observed_at) AS timeline_end,
    COUNT(DISTINCT date_trunc('HOUR', observed_at)) AS distinct_hours
FROM {SILVER_TABLE}
""").show(truncate=False)

print("=== Sample rows with observed_at ===")
spark.sql(f"""
SELECT robot_code, observed_at, abnormal_type, abnormal_score
FROM {SILVER_TABLE}
ORDER BY observed_at
LIMIT 10
""").show(truncate=False)

In [ ]:
# %% [Analysis A1] Load silver table → pandas for threshold analysis
# Goal: determine L1 (warning) and L2 (critical) thresholds on `abnormal_score`
#       such that NO robot triggers more than 10 times in 8000 cycles.
import pandas as pd

df_silver = spark.table(SILVER_TABLE).toPandas()
df_silver["observed_at"] = pd.to_datetime(df_silver["observed_at"])

print(f"Total rows       : {len(df_silver):,}")
print(f"Robots           : {sorted(df_silver['robot_code'].unique())}")
print(f"Date range       : {df_silver['observed_at'].min()}  →  {df_silver['observed_at'].max()}")
print()
print("=== Overall abnormal_score stats ===")
print(df_silver["abnormal_score"].describe(percentiles=[.5, .75, .9, .95, .99, .999]).to_string())
print()
print("=== severity_hint distribution ===")
print(df_silver["severity_hint"].value_counts().to_string())
print()
print("=== abnormal_type distribution ===")
print(df_silver["abnormal_type"].value_counts().to_string())
print()
print("=== Cycles per robot ===")
print(df_silver.groupby("robot_code").size().rename("cycles").to_string())

In [ ]:
# %% [Analysis A2] Score distribution plots per robot
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

robots = sorted(df_silver["robot_code"].unique())
n_robots = len(robots)
ncols = 3
nrows = (n_robots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = axes.flatten()

for i, robot in enumerate(robots):
    ax = axes[i]
    scores = df_silver[df_silver["robot_code"] == robot]["abnormal_score"]

    ax.hist(scores, bins=80, color="steelblue", edgecolor="white", alpha=0.85)

    for pct, color, ls in [
        (0.90, "gold",   "--"),
        (0.95, "orange", "--"),
        (0.99, "red",    "-"),
        (0.999,"darkred",":"),
    ]:
        v = scores.quantile(pct)
        ax.axvline(v, color=color, linestyle=ls, lw=1.2,
                   label=f"p{int(pct*100)}={v:.4f}")

    ax.set_title(f"{robot}  (n={len(scores):,})", fontsize=10)
    ax.set_xlabel("abnormal_score", fontsize=8)
    ax.set_ylabel("count", fontsize=8)
    ax.legend(fontsize=7)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Hide unused subplots
for j in range(n_robots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("abnormal_score Distribution per Robot\n(vertical lines = percentile reference points)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
display(fig)
plt.close()

# Also plot a combined CDF for a cross-robot view
fig2, ax2 = plt.subplots(figsize=(12, 5))
for robot in robots:
    scores = df_silver[df_silver["robot_code"] == robot]["abnormal_score"].sort_values()
    cdf = np.arange(1, len(scores) + 1) / len(scores)
    ax2.plot(scores.values, cdf, lw=1.5, label=robot)

ax2.set_xlabel("abnormal_score")
ax2.set_ylabel("CDF")
ax2.set_title("Cumulative Distribution of abnormal_score — all robots")
ax2.axhline(0.99, color="red", linestyle="--", lw=1, label="p99")
ax2.axhline(0.999, color="darkred", linestyle=":", lw=1, label="p99.9")
ax2.legend(fontsize=8)
plt.tight_layout()
display(fig2)
plt.close()

In [ ]:
# %% [Analysis A3] Percentile table per robot — tells us where the score mass sits
PCTS = [0.50, 0.75, 0.90, 0.95, 0.98, 0.99, 0.995, 0.999, 1.00]
PCT_LABELS = ["p50", "p75", "p90", "p95", "p98", "p99", "p99.5", "p99.9", "max"]

pct_rows = []
for robot in sorted(df_silver["robot_code"].unique()):
    scores = df_silver[df_silver["robot_code"] == robot]["abnormal_score"]
    row = {"robot": robot, "n": len(scores), "mean": round(scores.mean(), 6), "std": round(scores.std(), 6)}
    for p, lbl in zip(PCTS, PCT_LABELS):
        row[lbl] = round(float(scores.quantile(p)), 6)
    pct_rows.append(row)

pct_df = pd.DataFrame(pct_rows).set_index("robot")
print("=== Percentile table — abnormal_score ===")
print(pct_df.to_string())

# Show how many rows fall into each severity_hint bucket per robot
print("\n=== Rows per severity_hint per robot ===")
sev_pivot = df_silver.pivot_table(index="robot_code", columns="severity_hint", aggfunc="size", fill_value=0)
print(sev_pivot.to_string())

# Show is_file_anomaly breakdown
print("\n=== is_file_anomaly breakdown ===")
anom_pivot = df_silver.pivot_table(index="robot_code", columns="is_file_anomaly", aggfunc="size", fill_value=0)
anom_pivot.columns = ["normal", "anomaly"]
anom_pivot["anomaly_rate_%"] = (anom_pivot["anomaly"] / (anom_pivot["normal"] + anom_pivot["anomaly"]) * 100).round(3)
print(anom_pivot.to_string())

In [ ]:
# %% [Analysis A4] Sweep threshold grid — find where max robot triggers ≤ TARGET
TARGET_MAX_TRIGGERS = 10
TOTAL_CYCLES = 8000

# Fine-grained sweep: 0.00 → 1.00 in steps of 0.005
thresholds = np.round(np.arange(0.0, 1.001, 0.005), 4)

max_triggers_list = []
for t in thresholds:
    per_robot = df_silver[df_silver["abnormal_score"] >= t].groupby("robot_code").size()
    max_t = int(per_robot.max()) if len(per_robot) > 0 else 0
    max_triggers_list.append(max_t)

# Lowest threshold that still satisfies the ≤ TARGET constraint
valid = [(t, mx) for t, mx in zip(thresholds, max_triggers_list) if mx <= TARGET_MAX_TRIGGERS]
suggested_L1 = valid[0][0] if valid else None

# --- Plot: max triggers vs threshold ---
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(thresholds, max_triggers_list, color="steelblue", lw=2, label="Max triggers (worst robot)")
ax.axhline(TARGET_MAX_TRIGGERS, color="red", linestyle="--", lw=1.5, label=f"Target ≤ {TARGET_MAX_TRIGGERS}")
if suggested_L1 is not None:
    ax.axvline(suggested_L1, color="green", linestyle="--", lw=1.5, label=f"Suggested L1 = {suggested_L1}")
ax.set_xlabel("Threshold (abnormal_score ≥ threshold → trigger)")
ax.set_ylabel("Max trigger count across all robots")
ax.set_title("Threshold Sweep: Max Robot Triggers vs Score Threshold")
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
display(fig)
plt.close()

print(f"\nSuggested L1 threshold : {suggested_L1}")
print(f"  → max triggers at L1 : {max_triggers_list[list(thresholds).index(suggested_L1)] if suggested_L1 else 'N/A'}")

# --- Heatmap: triggers per robot across a coarser set of thresholds ---
coarse = np.round(np.arange(0.5, 1.001, 0.025), 3)
heat_data = {}
for t in coarse:
    per_robot = df_silver[df_silver["abnormal_score"] >= t].groupby("robot_code").size()
    heat_data[t] = per_robot

heat_df = pd.DataFrame(heat_data).fillna(0).astype(int)
heat_df.index.name = "robot"

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(heat_df.values, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_xticks(range(len(coarse)))
ax.set_xticklabels([str(t) for t in coarse], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index)
ax.set_xlabel("Threshold")
ax.set_ylabel("Robot")
ax.set_title("Trigger Count Heatmap: Robot × Threshold")
plt.colorbar(im, ax=ax, label="Trigger count")
# Annotate cells
for r in range(heat_df.shape[0]):
    for c in range(heat_df.shape[1]):
        val = heat_df.iloc[r, c]
        ax.text(c, r, str(val), ha="center", va="center", fontsize=7,
                color="white" if val > heat_df.values.max() * 0.6 else "black")
plt.tight_layout()
display(fig)
plt.close()

In [ ]:
# %% [Analysis A5] Detailed trigger counts at L1 / L2 candidates
# Produces the final comparison table before the recommendation cell.

L1_CANDIDATE = suggested_L1
# L2 is a tighter gate; start at L1 + 0.05, but clamp to [0, 1]
L2_CANDIDATE = round(min(suggested_L1 + 0.05, 1.0), 4) if suggested_L1 is not None else 0.95

print(f"Evaluating L1={L1_CANDIDATE}  L2={L2_CANDIDATE}\n")

summary_rows = []
for robot in sorted(df_silver["robot_code"].unique()):
    scores = df_silver[df_silver["robot_code"] == robot]["abnormal_score"]
    n = len(scores)
    l1_triggers = int((scores >= L1_CANDIDATE).sum())
    l2_triggers = int((scores >= L2_CANDIDATE).sum())
    summary_rows.append({
        "robot": robot,
        "total_cycles": n,
        f"triggers_L1>={L1_CANDIDATE}": l1_triggers,
        f"triggers_L2>={L2_CANDIDATE}": l2_triggers,
        "L1_rate_%": round(100 * l1_triggers / n, 4),
        "L2_rate_%": round(100 * l2_triggers / n, 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index("robot")
print(summary_df.to_string())

print(f"\n>>> L1={L1_CANDIDATE}: max triggers across robots = {summary_df[f'triggers_L1>={L1_CANDIDATE}'].max()}")
print(f">>> L2={L2_CANDIDATE}: max triggers across robots = {summary_df[f'triggers_L2>={L2_CANDIDATE}'].max()}")
print(f">>> Target: ≤ {TARGET_MAX_TRIGGERS} triggers per robot in {TOTAL_CYCLES} cycles")

# Bar chart: triggers per robot at L1 vs L2
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(summary_df))
w = 0.35
ax.bar(x - w/2, summary_df[f"triggers_L1>={L1_CANDIDATE}"], w, label=f"L1≥{L1_CANDIDATE}", color="steelblue")
ax.bar(x + w/2, summary_df[f"triggers_L2>={L2_CANDIDATE}"], w, label=f"L2≥{L2_CANDIDATE}", color="tomato")
ax.axhline(TARGET_MAX_TRIGGERS, color="black", linestyle="--", lw=1.5, label=f"Target ≤ {TARGET_MAX_TRIGGERS}")
ax.set_xticks(x)
ax.set_xticklabels(summary_df.index, rotation=30, ha="right")
ax.set_ylabel("Trigger count")
ax.set_title(f"Triggers per robot at L1={L1_CANDIDATE} and L2={L2_CANDIDATE}")
ax.legend()
plt.tight_layout()
display(fig)
plt.close()

In [ ]:
# %% [Analysis A6] Final recommendation printout
# Summarizes L1 / L2 candidates and prints a ready-to-copy config block

print("=" * 60)
print("RECOMMENDED THRESHOLD CONFIG")
print("=" * 60)
print(f"  L1 (Warning)  threshold : {L1_CANDIDATE}")
print(f"  L2 (Critical) threshold : {L2_CANDIDATE}")
print()
print(f"  Goal              : ≤ {TARGET_MAX_TRIGGERS} triggers / robot in {TOTAL_CYCLES} cycles")
print(f"  L1 worst robot    : {summary_df[f'triggers_L1>={L1_CANDIDATE}'].max()} triggers")
print(f"  L2 worst robot    : {summary_df[f'triggers_L2>={L2_CANDIDATE}'].max()} triggers")
print()
print("Copy-paste config block:")
print(f"""
ANOMALY_THRESHOLDS = {{
    "L1": {L1_CANDIDATE},   # Warning gate
    "L2": {L2_CANDIDATE},   # Critical / escalation gate
}}
""")

# Verify: show which specific rows would fire at L2 (the most actionable ones)
print("\n=== Sample L2 triggers (highest anomaly scores) ===")
display(
    spark.table(SILVER_TABLE)
         .filter(F.col("abnormal_score") >= L2_CANDIDATE)
         .orderBy(F.col("abnormal_score").desc())
         .limit(20)
)